<a href="https://colab.research.google.com/github/FlorianPerl/manipulative-speech-pl-mini/blob/main/manipulative-speech-pl-mini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Step 1: Install Dependencies

**Unsloth** is the core optimization layer — it rewrites PyTorch's attention kernels to run 2–5x faster with significantly less GPU memory. The remaining libraries (**peft, trl, transformers**) handle LoRA injection, the training loop, and model loading.

In [ ]:
# Unsloth installs itself with PyTorch — order matters here
!pip install unsloth

# --no-deps prevents pip from overwriting Unsloth's dependency versions
!pip install --no-deps accelerate peft trl transformers datasets

## Step 2: Load the Model and Tokenizer

Load a 4-bit version of Meta's Llama 3 8B.

4-bit quantization compresses the model weights from 32-bit floats to 4-bit integers, reducing VRAM usage from ~30GB to ~5GB with minimal accuracy loss — making it trainable on a Colab's free-tier T4 GPU.

In [ ]:
import torch
from unsloth import FastLanguageModel

max_seq_length = 2048  # Max number of tokens the model processes at once
load_in_4bit = True    # Compress weights to 4-bit to fit within T4 VRAM limits

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",  # Pre-quantized Llama 3 hosted by Unsloth
    max_seq_length = max_seq_length,
    dtype = None,        # Auto-detects optimal float type for the GPU (Float16 on T4)
    load_in_4bit = load_in_4bit,
)

## Step 3: Configure LoRA

LoRA injects small trainable matrices into specific layers of the frozen base model. Only these adapter matrices are updated during training, reducing trainable parameters from 8B to roughly 40 million (less than 1%).


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                        # (standard starting point) Adapter rank — higher means more capacity but more VRAM
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],  # Layers to inject adapters into, specific for Llama 3
    lora_alpha = 16,               # Scaling factor — equal to r for stable training
    lora_dropout = 0,              # (standard LoRA approach) Unsloth achieves better results without dropout
    bias = "none",                 # (standard LoRA approach) No bias terms added to adapter layers
    use_gradient_checkpointing = "unsloth",  # (standard for Unsloth) Trades compute for VRAM savings during backprop
    random_state = 3407,           # (arbitrary val) Seed for reproducibility
    use_rslora = False,            # (advanced LoRA)
    loftq_config = None,           # (advanced LoRA)
)
print("LoRA setup complete. Ready for training.")

## Step 4: Prepare Mini Dataset

Sefine a **prompt template** using Llama 3's native chat format and map the labeled Polish examples into it.

Each example is structured as a **system instruction**, a user-provided text, and the expected model response — the standard format for instruction fine-tuning.

In [ ]:
from datasets import Dataset

EOS_TOKEN = tokenizer.eos_token     # End-of-sequence token — tells the model when to stop generatin

def format_prompts(examples):
    texts = examples["text"]
    labels = examples["label"]
    outputs = []
    for text, label in zip(texts, labels):
        messages = [
            {"role": "system", "content": "Oceń, czy poniższy tekst zawiera techniki manipulacji, propagandy lub dezinformacji. Odpowiedz krótko: 'MANIPULACJA' lub 'NEUTRALNY', a następnie podaj jednozdaniowe uzasadnienie."},
            {"role": "user", "content": text},
            {"role": "assistant", "content": label},
        ]
        formatted = tokenizer.apply_chat_template(messages, tokenize=False) + EOS_TOKEN
        outputs.append(formatted)
    return {"text": outputs}

# 20 labeled examples — sufficient to validate the pipeline, not for real-world accuracy
mock_data = {
    "text": [
        # MANIPULACJA
        "Wszyscy prawdziwi Polacy doskonale wiedzą, że ten zbrodniczy traktat zniszczy naszą suwerenność i odda nas w ręce zagranicznych elit.",
        "Jeśli natychmiast nie zamkniemy granic, miliony nielegalnych imigrantów zaleją nasze ulice i odbiorą nam pracę.",
        "Skorumpowane media celowo milczą na ten temat, ponieważ boją się prawdy, którą tylko my mamy odwagę wam przekazać.",
        "Tylko ślepi zwolennicy obecnego rządu nie widzą, że te decyzje doprowadzą nasz kraj do absolutnej ruiny gospodarczej.",
        "Albo jesteś z nami, albo jesteś zdrajcą narodu — nie ma trzeciej drogi w tej walce o przyszłość Polski.",
        "Globalne elity od lat planują zniszczenie naszej tożsamości narodowej poprzez masową imigrację i narzucanie obcych wartości.",
        "Ci pseudo-eksperci sprzedali się zachodnim korporacjom i teraz kłamią nam w żywe oczy, żeby zniszczyć naszą gospodarkę.",
        "Mamy ostatnią szansę, żeby ocalić Polskę — jeśli nie zagłosujesz teraz, jutro może być za późno.",
        "Opozycja chce oddać Polskę Niemcom i Brukseli — to zdrada, za którą kiedyś odpowiedzą przed sądem historii.",
        "Lekarze popierający tę politykę są na liście płac wielkich korporacji farmaceutycznych i działają przeciwko zdrowiu Polaków.",
        # NEUTRALNY
        "Główny Urząd Statystyczny opublikował najnowszy raport dotyczący stopy bezrobocia w pierwszym kwartale bieżącego roku.",
        "Ministerstwo Edukacji Narodowej zapowiedziało wprowadzenie zmian w programie nauczania historii od przyszłego roku szkolnego.",
        "Narodowy Bank Polski utrzymał stopy procentowe na niezmienionym poziomie podczas ostatniego posiedzenia Rady Polityki Pieniężnej.",
        "Sąd Najwyższy wydał orzeczenie w sprawie interpretacji przepisów dotyczących prawa pracy w sektorze publicznym.",
        "Polska podpisała umowę o współpracy naukowo-technicznej z Europejską Agencją Kosmiczną na okres pięciu lat.",
        "Według danych GUS PKB Polski wzrósł w ubiegłym roku o 2,1 procent w stosunku do roku poprzedniego.",
        "Zarząd Dróg Miejskich ogłosił planowany remont odcinka ulicy Marszałkowskiej w Warszawie na przełomie maja i czerwca.",
        "Instytut Meteorologii i Gospodarki Wodnej wydał ostrzeżenie pierwszego stopnia przed silnym wiatrem dla województw pomorskiego i zachodniopomorskiego.",
        "Komisja Europejska zatwierdziła polskie plany dotyczące transformacji energetycznej na lata 2025–2035.",
        "Ministerstwo Zdrowia opublikowało kwartalne dane dotyczące liczby hospitalizacji z powodu chorób układu oddechowego.",
    ],
    "label": [
        # MANIPULACJA
        "MANIPULACJA (Użycie emocjonalnego języka, polaryzacja 'my vs oni', apel do strachu przed utratą suwerenności).",
        "MANIPULACJA (Straszenie katastroficzną wizją, wywoływanie paniki, nieuprawniona generalizacja).",
        "MANIPULACJA (Teoria spiskowa, dyskredytowanie mediów, budowanie syndromu oblężonej twierdzy).",
        "MANIPULACJA (Stygmatyzacja zwolenników rządu, katastrofizm, fałszywa pewność co do przyszłości).",
        "MANIPULACJA (Fałszywy dylemat — narzucenie tylko dwóch opcji, wykluczenie umiarkowanego stanowiska).",
        "MANIPULACJA (Teoria spiskowa o elitach, straszenie utratą tożsamości, nieuzasadniona generalizacja).",
        "MANIPULACJA (Ad hominem, dyskredytowanie ekspertów poprzez sugestię korupcji bez dowodów).",
        "MANIPULACJA (Fałszywa pilność, szantaż emocjonalny, manipulacja poprzez wywołanie strachu).",
        "MANIPULACJA (Demonizacja opozycji, użycie słowa 'zdrada' jako etykietki, apel do emocji nacjonalistycznych).",
        "MANIPULACJA (Dyskredytowanie środowiska medycznego przez nieudowodnione oskarżenia o korupcję).",
        # NEUTRALNY
        "NEUTRALNY (Suche przekazanie informacji opartej na danych instytucji publicznej).",
        "NEUTRALNY (Faktograficzny komunikat o planowanych zmianach administracyjnych).",
        "NEUTRALNY (Neutralne przekazanie decyzji instytucji finansowej bez oceny wartościującej).",
        "NEUTRALNY (Faktograficzne sprawozdanie z orzeczenia sądowego bez komentarza redakcyjnego).",
        "NEUTRALNY (Informacja o umowie międzynarodowej podana w formie suchego komunikatu).",
        "NEUTRALNY (Dane statystyczne przedstawione bez manipulacji kontekstem).",
        "NEUTRALNY (Komunikat administracyjny o planowanych pracach infrastrukturalnych).",
        "NEUTRALNY (Ostrzeżenie meteorologiczne oparte na danych naukowych, bez elementów emocjonalnych).",
        "NEUTRALNY (Neutralna informacja o decyzji instytucji unijnej, bez komentarza wartościującego).",
        "NEUTRALNY (Przekazanie danych statystycznych ministerstwa bez interpretacji politycznej).",
    ]
}

dataset = Dataset.from_dict(mock_data)
dataset = dataset.map(format_prompts, batched=True)   # Apply formatting across all examples at once

## Step 5: Execute Training

Pass the formatted dataset to Hugging Face's **SFTTrainer** (Supervised Fine-Tuning Trainer), which runs the PyTorch training loop and adjusts the LoRA adapter weights. All training hyperparameters are defined in **SFTConfig**.

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",       # Formatted prompts
        max_seq_length = max_seq_length,   # Truncate any sequence longer than 2048 tokens
        per_device_train_batch_size = 2,   # (value standard for Colab's T4 GPU) Number of examples processed per GPU step
        gradient_accumulation_steps = 4,   # (standard) Accumulate gradients over 4 steps before updating weights — simulates a batch size of 8
        warmup_steps = 1,                  # (applicable in small projects - larger-size projects could use 10-100 steps) Gradually ramp up learning rate to avoid unstable early updates
        max_steps = 6,                     # Total training steps — low for PoC purposes
                                              # For a real project, max_steps is typically replaced by num_train_epochs — the number of full passes through the entire dataset:
        learning_rate = 2e-4,              # (standard for LoRA) Step size for weight updates
        fp16 = not torch.cuda.is_bf16_supported(),  # Use 16-bit floats if BF16 unavailable
        bf16 = torch.cuda.is_bf16_supported(),      # BF16 preferred on modern GPUs — more numerically stable than FP16
        logging_steps = 1,                 # Log training loss after every step
        output_dir = "outputs",            # Dir for temp training checkpoints
    ),
)

trainer_stats = trainer.train()

# Persist the fine-tuned adapter weights (not the full model)
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")
print("Adapter saved to ./lora_model")

## Step 6: Test the fine-tuned model

Switch the model from training mode to **inference mode** and pass a completely unseen Polish text through it.

In [22]:
FastLanguageModel.for_inference(model)    # Switches off training-specific operations (e.g. gradient tracking)

#test_text = "Tylko ślepi zwolennicy obecnego rządu nie widzą, że te decyzje doprowadzą nasz kraj do absolutnej ruiny gospodarczej."
#test_text = "Czy prawdziwi Polacy widzą zagrożenie ze strony Brukseli?"
#test_text = "Poddani Tuska z wizytą w Parlamencie Europejskim"
#test_text = "Ideologia LGBTQ+: bezpieczna dla naszych dzieci?"
test_text = "LGBTQ+: świadoma decyzja"

messages = [
    {"role": "system", "content": "Oceń, czy poniższy tekst zawiera techniki manipulacji, propagandy lub dezinformacji. Odpowiedz krótko: 'MANIPULACJA' lub 'NEUTRALNY', a następnie podaj jednozdaniowe uzasadnienie."},
    {"role": "user", "content": test_text},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,               # Convert text to token IDs immediately
    add_generation_prompt = True,  # Appends the assistant header so the model knows to respond
    return_tensors = "pt"          # Return as PyTorch tensors
).to("cuda")                       # Move to GPU memory

outputs = model.generate(
    input_ids = inputs,
    max_new_tokens = 64,                        # Maximum tokens the model can generate in response
    use_cache = True,                           # Reuse previously computed states for faster generation
    pad_token_id = tokenizer.eos_token_id,      # Prevents warnings when sequences have different lengths
)

response = tokenizer.batch_decode(outputs, skip_special_tokens=True)    # Token IDs back to text
parts = response[0].split("assistant")

print("\n--- TEST INPUT ---")
print(test_text)
print("\n--- MODEL OUTPUT ---")
print(parts[-1].strip() if len(parts) > 1 else response[0])             # Print only the model's response

Both `max_new_tokens` (=64) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- TEST INPUT ---
LGBTQ+

--- MODEL OUTPUT ---
NEUTRALNY - Termin LGBT+ jest neutralnym określeniem grupy społecznej, nie zawierającym manipulacji.
